# Mistral predictions

In [9]:
# !pip install mistralai

In [ ]:
from mistralai import Mistral
api_key = os.environ["MISTRAL_API_KEY"]
model = "mistral-small-2409"

client = Mistral(api_key=api_key)

In [ ]:
df = pd.read_csv(ANNOTATED_SAMPLE)

# Embeddings

In [13]:
# model = "mistral-embed"

# client = Mistral(api_key=api_key)

# embeddings_batch_response = client.embeddings.create(
#     model=model,
#     inputs=df['message'],
# )

# len(df)
# len(embeddings_batch_response.data)
# len(embeddings_batch_response.data[0].embedding) # embedding dimension = 1024
# embeddings = [item.embedding for item in embeddings_batch_response.data]

In [14]:
# import numpy as np
# from sklearn.neighbors import NearestNeighbors

# def get_NN(embeddings, n, k=1):
#   # Convert to NumPy array
#   X = np.array(embeddings)

#   # Fit kNN (using cosine similarity, common for embeddings)
#   knn = NearestNeighbors(n_neighbors=k, metric='cosine')
#   knn.fit(X)

#   # Query vector
#   query_vec = embeddings_batch_response.data[n].embedding

#   # Convert to array and reshape
#   query_vec = np.array(query_vec).reshape(1, -1)

#   # Find nearest neighbors
#   distances, indices = knn.kneighbors(query_vec)

#   return indices.flatten() # Return only the flattened indices

## Few-shot RAG with completion agent

In [ ]:
df_full = pd.read_csv(FULL_MESSAGES)
df_train = pd.read_csv(ANNOTATED_SAMPLE)
df_train = df_train[df_train['topic_category'].notna()]

C:\Users\167266\AppData\Local\Temp\ipykernel_22024\2648094082.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv(r"F:\._PhD\Publications\SocSem_Telegram\Revisions\Code\seed_walther_clean.csv")


In [7]:
import os

from mistralai import Mistral
import numpy as np

# Initialize the client
client = Mistral(api_key)

def classify_text_mistral_agent_RAG(text, df_train, train_embeddings, knn_model):

  # Calculate embedding for the current text
  current_embedding_response = client.embeddings.create(
      model="mistral-embed",
      inputs=[text],
  )
  current_embedding = current_embedding_response.data[0].embedding
  current_embedding_array = np.array(current_embedding).reshape(1, -1)

  # Find the nearest neighbor in the training embeddings
  distances, indices = knn_model.kneighbors(current_embedding_array)
  nearest_neighbor_index_in_train = indices.flatten()[0]

  # Get the example message and categories from the training set
  example_message = df_train.iloc[nearest_neighbor_index_in_train]['message']
  example_categories = df_train.iloc[nearest_neighbor_index_in_train]['topic_category_3']


  # Define the messages
  inputs = [
        {"role": "user", "content": f"""You are a social media text classifier. Read each message carefully and use the example to classify it into one or more of the following categories:

    1. "UK politics and Brexit": general commentary of British news and politics, including Brexit.
    2. "Online coordination": messages about online communications, including but not restricted to content promotion and live session announcements.
    3. "Nationalism and white supremacy": messages defending the absolute superiority of the white race and Western countries (especially Europe and North America), leveraging arguments from demographics, history, religion, Hitler and Nazism.
    4. "Immigration": messages mentioning migrants, usually hostile to migrants but without a specific target mentioned.
    5. "Anti-left": criticisms of left-wing parties and activist groups.
    6. "US politics and Trumpism": general commentary of US news and politics, including anything related to Donald Trump.
    7. "Offline activism, mobilization and violence": messages commenting on or calling for offline street protest, including comments on street violence.
    8. "Anti-establishment, lies and disinformation": criticism of mainstream media, legacy politics or any other group, including accusations of intentionally lying or hiding the truth. This includes conspiracy theories.
    9. "Proud Boys/Antifa and the manosphere": messages by or about the Proud Boys or the manosphere. These should feature misogyny, homophobia, transphobia, mentions of Tommy Robinson, criticisms of the Antifa, and red/black/blue pill references.
    10. "Antisemitism and islamophobia": hateful comments on Jews/Israel or Muslims.
    11. "None of the above": messages about any other topic than those listed above.

   Return your output strictly as a Python list of category names without any additional text.

    Example message: {example_message}
    Example categories: {example_categories}


    Message: {text}

    Categories: """}
  ]

  # Define completion arguments
  completion_args = {
      "temperature": 0,
      "max_tokens": 2048,
      "top_p": 1
  }

  try:
      # Make the API call
      response = client.beta.conversations.start_stream(
          inputs=inputs,
          model="mistral-large-2411",
          instructions="""""",
          completion_args=completion_args,
      )

      # Process the stream
      full_response = ""
      for chunk in response:
        if chunk.event == 'message.output.delta':
          full_response += chunk.data.content
  except Exception as e:
      print(f"Error: {str(e)}")

  return full_response

In [8]:
import time
import random
import os
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
import numpy as np
import pandas as pd
from mistralai import Mistral

# Adjustable parameters
POLITE_DELAY = 0.5
MAX_ATTEMPTS = 6
BASE_BACKOFF = 1.5
CHECKPOINT_EVERY = 50
CHECKPOINT_FILE = "checkpoint_mistral_predictions.csv"

# Initialize client
client = Mistral(api_key=api_key)

# --- Helper: Retry wrapper with exponential backoff ---
def retry_with_backoff(func, *args, max_attempts=MAX_ATTEMPTS, base_delay=BASE_BACKOFF, **kwargs):
    attempt = 0
    while True:
        try:
            return func(*args, **kwargs)
        except Exception as e:
            attempt += 1
            if attempt >= max_attempts:
                print(f"[error] Failed after {attempt} attempts: {e}")
                raise
            backoff = base_delay * (2 ** (attempt - 1))
            jitter = random.uniform(0, backoff * 0.3)
            sleep_time = backoff + jitter
            print(f"[warning] Attempt {attempt}/{max_attempts} failed ({e}). Retrying in {sleep_time:.1f}s...")
            time.sleep(sleep_time)


# --- Step 1: Compute or load train embeddings ---
if 'embeddings_train' not in locals():
    print("Computing training embeddings...")
    try:
        embeddings_batch_response = retry_with_backoff(
            client.embeddings.create,
            model= "mistral-embed",
            inputs=df_train['message'].tolist()
        )
        embeddings_train = [item.embedding for item in embeddings_batch_response.data]
    except Exception as e:
        print(f"[fatal] Could not compute train embeddings: {e}")
        raise

# --- Step 2: Fit KNN model ---
knn = NearestNeighbors(n_neighbors=1, metric='cosine')
knn.fit(np.array(embeddings_train))

# --- Step 3: Resume logic ---
if os.path.exists(CHECKPOINT_FILE):
    print(f"[resume] Found checkpoint: {CHECKPOINT_FILE}")
    df_full_checkpoint = pd.read_csv(CHECKPOINT_FILE)

    # If the new df_full differs (e.g. new rows), align by index
    if len(df_full_checkpoint) < len(df_full):
        print(f"[resume] Checkpoint has {len(df_full_checkpoint)} rows, current data has {len(df_full)}. Extending...")
        df_full.loc[:len(df_full_checkpoint)-1, 'mistral_agent_predictions'] = df_full_checkpoint.get('mistral_agent_predictions', "")
    else:
        df_full = df_full_checkpoint.copy()

    # Determine where to resume
    processed = df_full['mistral_agent_predictions'].notna()
    start_index = processed.sum()
    print(f"[resume] Resuming from message {start_index + 1}/{len(df_full)}")
else:
    df_full['mistral_agent_predictions'] = np.nan
    start_index = 0
    print("[start] No checkpoint found, starting from scratch.")

# --- Step 4: Classification loop with resume ---
for i in tqdm(range(start_index, len(df_full)), desc="Classifying messages", initial=start_index, total=len(df_full)):
    msg = df_full.loc[i, 'message']

    # Retry classification
    try:
        result = retry_with_backoff(
            classify_text_mistral_agent_RAG,
            msg, df_train, embeddings_train, knn
        )
    except Exception as e:
        print(f"[error] Skipping message {i} after repeated failure: {e}")
        result = ""

    df_full.at[i, 'mistral_agent_predictions'] = result

    # Checkpoint every N messages
    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(df_full):
        df_full.to_csv(CHECKPOINT_FILE, index=False)
        print(f"[checkpoint] Saved progress up to message {i + 1}/{len(df_full)}")

    time.sleep(POLITE_DELAY)

print("✅ Classification complete.")
df_full.to_csv("final_mistral_predictions.csv", index=False)
print("💾 Saved final results to 'final_mistral_predictions.csv'.")


Computing training embeddings...
[resume] Found checkpoint: checkpoint_mistral_predictions.csv


C:\Users\167266\AppData\Local\Temp\ipykernel_22024\2811268637.py:59: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full_checkpoint = pd.read_csv(CHECKPOINT_FILE)


[resume] Resuming from message 50912/61391


Classifying messages:  83%|████████▎ | 50912/61391 [00:01<4:53:47,  1.68s/it]

[warning] Attempt 1/6 failed (API error occurred: Status 429. Body: {"object":"error","message":"Service tier capacity exceeded for this model.","type":"service_tier_capacity_exceeded","param":null,"code":"3505"}). Retrying in 1.7s...


Classifying messages:  83%|████████▎ | 50914/61391 [00:16<16:47:19,  5.77s/it]

... (1,475 outputs left out)

Classifying messages: 100%|█████████▉| 61390/61391 [8:14:31<00:07,  7.57s/it]

[checkpoint] Saved progress up to message 61391/61391


Classifying messages: 100%|██████████| 61391/61391 [8:14:46<00:00,  2.83s/it]


✅ Classification complete.
💾 Saved final results to 'final_mistral_predictions.csv'.


## Performance metrics

In [ ]:
df_manual = pd.read_csv(ANNOTATED_SAMPLE)
df_mistral = pd.read_csv(MISTRAL_PREDICTIONS)

C:\Users\167266\AppData\Local\Temp\ipykernel_25436\1401799544.py:2: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mistral = pd.read_csv(r"F:\._PhD\Publications\SocSem_Telegram\Revisions\Code\final_mistral_predictions.csv")


In [84]:
df_eval = pd.merge(df_manual, df_mistral[['message','mistral_agent_predictions']], on='message', how='inner').drop_duplicates(subset=['message'])
len(df_eval)

222

In [ ]:
# Clean predicted labels
import re
import ast
# --- Fix incomplete labels ---

label_lookup = {
    "['Proud Boys/Ant": "['Proud Boys/Antifa and the manosphere']",
    "['Proud Boys/Antifa and the manosphere', 'UK politics and Brexit', 'Offline activism, mobilization and violence', 'Nationalism" : "['Proud Boys/Antifa and the manosphere', 'UK politics and Brexit', 'Nationalism']",
    "['None of" : "['None of the above']",
    "['UK politics and Brexit', 'Offline activism, mobilization and violence', 'Anti-establishment, lies and": "['UK politics and Brexit', 'Anti-establishment, lies and disinformation']"
}

def fix_incomplete_label(label, lookup):
    if not isinstance(label, str):
        return label
    for prefix, full_label in lookup.items():
        # Check if the label starts with the prefix (case-insensitive)
        if label.strip().lower().startswith(prefix.lower()):
            return full_label
    return label
df_eval['mistral_agent_predictions'] = df_eval['mistral_agent_predictions'].apply(lambda lbl: fix_incomplete_label(lbl, label_lookup))


# --- Extract labels ---
def extract_labels(x):
    # Convert to string and replace double quotes with single quotes
    x = str(x).replace('"', "'")
    
    # Extract content that looks like a Python list of strings, e.g. "['label1', 'label2']"
    match = re.search(r"\[([^\[\]]*'[^]]*?)\]", x)
    if match:
        try:
            labels = ast.literal_eval("[" + match.group(1) + "]")
            # Keep only strings, strip whitespace
            labels = [lbl.strip() for lbl in labels if isinstance(lbl, str)]
            # Remove duplicates while preserving order
            seen = set()
            labels = [x for x in labels if not (x in seen or seen.add(x))]
            return labels
        except Exception:
            pass
    return []

df_eval['clean_labels'] = df_eval['mistral_agent_predictions'].apply(extract_labels)

# --- Reset index ---
df_eval = df_eval[df_eval['clean_labels'].notna()]
df_eval = df_eval.reset_index(drop=True)

# --- Print label distribution table ---
df_eval['clean_labels'].value_counts(normalize=True,dropna=False)

clean_labels
[None of the above]                                                                                                                                    0.328829
[Proud Boys/Antifa and the manosphere]                                                                                                                 0.112613
[Online coordination]                                                                                                                                  0.067568
[Nationalism and white supremacy]                                                                                                                      0.058559
[US politics and Trumpism, Online coordination]                                                                                                        0.036036
[UK politics and Brexit]                                                                                                                               0.036036
[Anti-establishment, lies a

In [ ]:
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, recall_score, precision_score
from sklearn.preprocessing import MultiLabelBinarizer

import ast

def safe_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    elif isinstance(x, list):
        return x
    else:
        return []
        
df_eval['topic_category'] = df_eval['topic_category'].apply(safe_eval)



mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df_eval['topic_category_3'])
y_pred = mlb.transform(df_eval['clean_labels'].apply(
    lambda x: ['None of the above'] if x is None or (isinstance(x, float) and pd.isna(x)) else x
))
# y_pred = mlb.fit_transform(df_new['geminiflashlite_predictions'])

# Precision
precision = precision_score(y_true, y_pred, average="micro")

# Recall
recall = recall_score(y_true, y_pred, average='micro')

# Micro F1-score (recommended)
f1_micro = f1_score(y_true, y_pred, average='micro')

# Macro F1-score
f1_macro = f1_score(y_true, y_pred, average='macro')

# Hamming loss
hamming = hamming_loss(y_true, y_pred)

# Subset accuracy
subset_acc = accuracy_score(y_true, y_pred)

print(f"Precision score: {precision:.3f}")
print(f"Recall score: {recall:.3f}")
print(f"Micro F1-score: {f1_micro:.3f}")
print(f"Macro F1-score: {f1_macro:.3f}")
print(f"Hamming Loss: {hamming:.3f}")
print(f"Subset Accuracy: {subset_acc:.3f}")

Precision score: 0.817
Recall score: 0.929
Micro F1-score: 0.870
Macro F1-score: 0.728
Hamming Loss: 0.026
Subset Accuracy: 0.743
